In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/obb-dataset-co2/custom_merged_obb/data.yaml
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/train.cache
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid.cache
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/Door2765_png.rf.841d21462ed873632a60ecf0695f7201.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/2611_jpg.rf.a080f9fc31f30e8e82d6a1f263a6fa1f.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/06030-928120060232-2-LGA_jpg.rf.8fdca063142a02b3db12ccefcfc85d93.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/01163_jpg.rf.ccc20a810964b74d2d7b9e084aebc101.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/Door0091_png.rf.2248152cc975ca056be17735ac52e846.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/2490_jpg.rf.fe5837d247f64e14679ada87bf412afd.txt
/kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid/Door0679_png.rf.d6fcc3947bf0fabf66583e2c3e7fe37d.txt
/kagg

In [2]:
!pip install ultralytics==8.3.209

from ultralytics import YOLO
import torch, shutil

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

WARNING ⚠️ Ultralytics settings reset to default values. This may be due to a possible problem with your settings or a recent ultralytics package update. 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
GPU available: True
GPU: Tesla P100-PCIE-16GB


In [3]:
zip_path = "/kaggle/input/obb-dataset-co2/custom_merged_obb.zip"
extract_path = "/kaggle/input/obb-dataset-co2/custom_merged_obb"

if os.path.exists(zip_path):
    print("Extracting dataset...")
    shutil.unpack_archive(zip_path, extract_path)
else:
    print("Dataset already extracted.")

print("Dataset path:", extract_path)

Dataset already extracted.
Dataset path: /kaggle/input/obb-dataset-co2/custom_merged_obb


In [4]:


yaml_path = "/kaggle/input/obb-dataset-co2/custom_merged_obb/data.yaml"

# Just print and visually confirm
!cat {yaml_path}

# Optionally check that the train/valid folders exist
print("\nChecking folders:")
!ls /kaggle/input/obb-dataset-co2/custom_merged_obb/images
!ls /kaggle/input/obb-dataset-co2/custom_merged_obb/images/valid | head -n 5


# train: E:/sem7/FYP/9_24/objectdetetction/datasets/custom_merged_obb/images/train
# val: E:/sem7/FYP/9_24/objectdetetction/datasets/custom_merged_obb/images/valid

# names:
#   0: fire_extinguisher
#   1: switch_panel
#   2: door
#   3: gauge
#   4: person


# data.yaml for OBB YOLO11 (Kaggle version)

path: /kaggle/input/obb-dataset-co2/custom_merged_obb  # root dataset folder (update to your uploaded dataset name)
train: images/train
val: images/valid

# number of classes
nc: 4        # example – change to your actual number of classes

# class names
names:
  0: fire_extinguisher
  1: switch_panel
  2: door
  3: gague


Checking folders:
train  valid
00002_jpg.rf.269e7687a70b74aab01c2865037fb870.jpg
00011-418319060016-1-FINAL2_jpg.rf.62f28fbb3aa0423840d89370d8b302c7.jpg
00011-418319060016-1-FINAL3_jpg.rf.89f3db1594957449bcd70c255d53332b.jpg
00017_jpg.rf.a8d2aefe6e0729ff80e5f3fc50c52a17.jpg
00020-418319100144-1-FINAL2_jpg.rf.e517edace41907ec6c766dcb19205689.jpg
ls: write error: Broke

In [5]:
from ultralytics.data.utils import check_det_dataset

print("Checking dataset integrity...")
dataset_info = check_det_dataset(yaml_path)
print("Dataset verified successfully!")
print(f"Train: {dataset_info['train']}")
print(f"Val:   {dataset_info.get('val', 'N/A')}")
print(f"Classes: {dataset_info['names']}")

Checking dataset integrity...
Dataset verified successfully!
Train: /kaggle/input/obb-dataset-co2/custom_merged_obb/images/train
Val:   /kaggle/input/obb-dataset-co2/custom_merged_obb/images/valid
Classes: {0: 'fire_extinguisher', 1: 'switch_panel', 2: 'door', 3: 'gague'}


In [6]:
import os

base = "/kaggle/input/obb-dataset-co2/custom_merged_obb"
for sub in ["images/train", "images/valid", "labels/train", "labels/valid"]:
    path = os.path.join(base, sub)
    print(f"\n🔍 Checking {path}")
    print("Count:", len(os.listdir(path)))
    bad = [f for f in os.listdir(path) if os.path.getsize(os.path.join(path, f)) == 0]
    if bad:
        print("Empty files:", bad[:5])
    else:
        print("No empty files found")



🔍 Checking /kaggle/input/obb-dataset-co2/custom_merged_obb/images/train
Count: 7001
No empty files found

🔍 Checking /kaggle/input/obb-dataset-co2/custom_merged_obb/images/valid
Count: 1751
No empty files found

🔍 Checking /kaggle/input/obb-dataset-co2/custom_merged_obb/labels/train
Count: 7001
Empty files: ['image-14_jpg.rf.bb0a30b951b4a2e0ecb1bb32d27eb75d.txt', 'image_72_jpg.rf.7a5729f9a6123032ee2ac0a7a590091a.txt', 'frame_384_jpg.rf.8a05d86b7c141ffe0f3ffe6e2fda174d.txt', 'image-18_jpg.rf.a58cc18fa5f73e5e132c269a37f4de4a.txt', 'image_43_jpg.rf.8e92d072df380b9aab4f87d05c47cb79.txt']

🔍 Checking /kaggle/input/obb-dataset-co2/custom_merged_obb/labels/valid
Count: 1751
Empty files: ['image_19_jpg.rf.c27d3b33a7cada830c4aabe49151bf57.txt', 'image_27_jpg.rf.d568009a70a684228293b93071af0b44.txt', 'image_46_jpg.rf.f48cfd2f5ffb137e56956a777fe8f0a2.txt']


In [7]:
import shutil, os

src = "/kaggle/input/obb-dataset-co2/custom_merged_obb"
dst = "/kaggle/working/custom_merged_obb"

# copy entire dataset tree (takes ~1-2 min)
if not os.path.exists(dst):
    shutil.copytree(src, dst)
    print("Dataset copied to /kaggle/working/")
else:
    print("Dataset already exists in /kaggle/working/")


Dataset already exists in /kaggle/working/


In [8]:
# clean the copy in /kaggle/working/
def remove_empty_labels(base_path):
    count_removed = 0
    for sub in ["labels/train", "labels/valid"]:
        folder = os.path.join(base_path, sub)
        for f in os.listdir(folder):
            path = os.path.join(folder, f)
            if os.path.getsize(path) == 0:
                os.remove(path)
                img_name = f.replace(".txt", ".jpg")
                for img_dir in ["images/train", "images/valid"]:
                    img_path = os.path.join(base_path, img_dir, img_name)
                    if os.path.exists(img_path):
                        os.remove(img_path)
                count_removed += 1
    print(f"✅ Removed {count_removed} empty label files and their images.")

remove_empty_labels("/kaggle/working/custom_merged_obb")


✅ Removed 0 empty label files and their images.


In [9]:

import sys, types, os

# Disable external loggers
os.environ["WANDB_MODE"] = "disabled"
os.environ["WANDB_DISABLED"] = "true"
os.environ["RAY_AIR_NEW_OUTPUT"] = "0"

# Create dummy ray modules so any 'import ray.train._internal.session' resolves safely
ray = types.ModuleType("ray")
train = types.ModuleType("train")
_internal = types.ModuleType("_internal")
session = types.ModuleType("session")
session._get_session = lambda: None   # <- the problematic call returns None harmlessly
_internal.session = session
train._internal = _internal
ray.train = train
sys.modules["ray"] = ray
sys.modules["ray.train"] = train
sys.modules["ray.train._internal"] = _internal
sys.modules["ray.train._internal.session"] = session

print("Ray hard-blocked & W&B disabled")


Ray hard-blocked & W&B disabled


In [10]:
yaml_path = "/kaggle/working/custom_merged_obb/data.yaml"
with open(yaml_path, "r") as f:
    txt = f.read()
txt = txt.replace("/kaggle/input/obb-dataset-co2/custom_merged_obb", "/kaggle/working/custom_merged_obb")
with open(yaml_path, "w") as f:
    f.write(txt)
print("data.yaml now points to /kaggle/working/custom_merged_obb")


data.yaml now points to /kaggle/working/custom_merged_obb


In [11]:
import ultralytics
print(ultralytics.__version__)


8.3.209


In [13]:
from ultralytics import YOLO

model = YOLO("yolo11n-obb.pt")
results = model.train(
    data="/kaggle/working/custom_merged_obb/data.yaml",
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=0,
    cache=False,
    project="obb_run",
    name="train_final_fixed",
    save_period=5,
    patience=20,
    lr0=0.01,
    augment=True
)


Ultralytics 8.3.209 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/custom_merged_obb/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-obb.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train_final_fixed5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

KeyboardInterrupt: 

In [15]:
import os
os.listdir("/kaggle/working/obb_run/train_final_fixed4/weights")


['epoch50.pt',
 'epoch25.pt',
 'epoch0.pt',
 'epoch35.pt',
 'epoch40.pt',
 'epoch30.pt',
 'epoch45.pt',
 'last.pt',
 'epoch5.pt',
 'epoch20.pt',
 'best.pt',
 'epoch15.pt',
 'epoch10.pt']

In [16]:
import os, shutil

final_path = "/kaggle/working/final_weights"
os.makedirs(final_path, exist_ok=True)

src_folder = "/kaggle/working/obb_run/train_final_fixed4/weights"

for w in ["best.pt", "last.pt"]:
    src = os.path.join(src_folder, w)
    if os.path.exists(src):
        shutil.copy(src, final_path)
        print(f"Copied {w} → {final_path}")

print("Check /kaggle/working/final_weights for saved models.")


Copied best.pt → /kaggle/working/final_weights
Copied last.pt → /kaggle/working/final_weights
Check /kaggle/working/final_weights for saved models.


In [ ]:
from ultralytics import YOLO

# Load the best model
model = YOLO("/kaggle/working/obb_run/train_final_fixed4/weights/best.pt")

# Run validation on the same validation split defined in your data.yaml
metrics = model.val(data="/kaggle/working/custom_merged_obb/data.yaml")

# Print summary
print(metrics)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load training log
df = pd.read_csv("/kaggle/working/obb_run/train_final_fixed4/results.csv")

# Plot loss curves
plt.figure()
plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss")
plt.plot(df["epoch"], df["train/cls_loss"], label="Cls Loss")
plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curves")
plt.legend()
plt.grid(True)
plt.show()

# Plot mAP curves
plt.figure()
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@0.5")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@0.5:0.95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation mAP Progress")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Visualize a few validation images with predicted rotated boxes
model.predict(
    source="/kaggle/working/custom_merged_obb/images/valid",
    conf=0.5,
    save=True,
    project="obb_run",
    name="val_predictions"
)
